# PCA on the GLM input

Run this notebook from its import cell to the end in a clean kernel. It uses the same preprocessed fMRI run and numerical design matrix as the GLM.

The analysis treats one timepoint as one sample, selects and displays spatial loadings separately in axial view, and correlates the first ten temporal scores with the GLM regressors. Files are read from and saved to the shared tsclient drive using the Part 1 directory convention.


In [ ]:
# Import the libraries used for PCA and the GLM comparison
import os
import os.path as op
import json
import pandas as pd
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
from nilearn.plotting import plot_stat_map
from sklearn.decomposition import PCA

# Project directories
notebook_dir = os.path.abspath("")
parent_dir = os.path.abspath(op.join(notebook_dir, '..'))
data_dir = '/home/jovyan/data/tsclient/NSSP/progetto'
output_dir = op.join(data_dir, 'outputs')

# Alternative VM paths
#data_dir = '/home/jovyan/data/tsclient/Projects/Project_1/miniproject_2026_final'
#output_dir = '/home/jovyan/data/tsclient/Projects/Project_1/Outputs'

# Store derivatives on the shared drive for persistence
shared_dir = op.join(output_dir, 'shared_glm_pca')
save_results = op.join(output_dir, 'pca')
if not op.isdir(data_dir):
    raise FileNotFoundError('Dataset directory not accessible: ' + data_dir)
os.makedirs(save_results, exist_ok=True)

print('Notebook directory:', notebook_dir)
print('Project directory:', parent_dir)
print('Data directory:', data_dir)
print('Output directory:', output_dir)
print('Shared GLM files:', shared_dir)
print('PCA results:', save_results)

%matplotlib inline

PCA finds orthogonal spatial directions ordered by explained variance. We load the fMRI run, inspect a foreground mask, form a voxel-by-timepoint matrix, subtract each voxel's temporal mean and fit PCA on its transpose. Thus one timepoint is one sample and each masked voxel is one feature.

`samples = data[slice_non_background]` has shape V x T, so `mean(axis=1)` computes temporal means. We do not subtract each volume's spatial mean or standardize voxel variances. Scikit-learn also centers features internally.

Spatial loadings are stored in `pca.components_`. Projecting with `pca.transform(X.T)` gives temporal scores. GLM regressors are used only to interpret these scores after fitting PCA.


## Part 1: Apply PCA with one timepoint as one sample

### Load the shared fMRI run

Run this notebook from its import cell in a clean kernel in the VM. Inputs and persistent derivatives follow the Part 1 shared-drive convention: `data_dir` points to `/home/jovyan/data/tsclient/NSSP/progetto`, `output_dir` is `data_dir/outputs`, and PCA results are saved in `output_dir/pca`. The commented paths in the setup cell provide an alternative VM directory layout. Before running, make `shared_dir` agree with the actual GLM export location; the existing GLM exporter may use `~/NSSP_outputs/shared_glm_pca` instead. The exported design and task-regressor metadata must describe the same selected GLM design. These upstream exports are prerequisites, not regenerated by the PCA fit.

Use the same already-trimmed, motion-corrected and 6 mm-smoothed series used by the GLM. The GLM must export `design_matrix.csv`, indexed by `frame_time_s`, and `metadata.json` into `output_dir/shared_glm_pca`. `design_matrix_metadata.json` is also accepted as the metadata filename. The GLM export instructions below use the same shared-drive directory.

The metadata identify the input image, TR, removed initial volumes, retained volume count, task regressors and design columns. The image path must identify the actual preprocessed file used by the GLM and be readable inside the VM. Keep that input among the persistent Part 1 derivatives; regenerate the export if the file is moved. Do not remove the initial volume again or shift the events a second time. Spike regressors model outlier volumes without removing them from this series.

We reuse all nonconstant regressors actually exported by the GLM, including task sub-events if present, motion, cosine drift and outlier spikes. Overlapping task regressors can have similar temporal associations; marginal correlations do not separate their unique effects. The 6 mm smoothing is retained for consistency with the GLM, although it reduces spatial detail. These files are sufficient for temporal correlations; a fitted GLM is needed only for the later comparison of spatial maps.

In [ ]:
# Read the numerical design and the metadata for the same fMRI run
design_path = op.join(shared_dir, 'design_matrix.csv')
metadata_path = op.join(shared_dir, 'metadata.json')
if not op.isfile(metadata_path):
    metadata_path = op.join(shared_dir, 'design_matrix_metadata.json')
if not (op.isfile(design_path) and op.isfile(metadata_path)):
    raise FileNotFoundError(
        'Export design_matrix.csv and metadata.json from the GLM to '
        + shared_dir + ', or set shared_dir to the export folder.'
    )

with open(metadata_path, 'r', encoding='utf-8') as f:
    metadata = json.load(f)
design_mat = pd.read_csv(design_path, index_col='frame_time_s')

img = nib.load(metadata['fmri_path'])
assert len(img.shape) == 4, 'PCA requires a 4D fMRI time series.'
affine = img.affine

# Both export formats describe the same image and design conventions
tr = float(metadata['tr_s'] if 'tr_s' in metadata else metadata['tr_seconds'])
design_columns = metadata['columns'] if 'columns' in metadata else metadata['design_columns']
task_regressors = metadata['task_regressors'] if 'task_regressors' in metadata else metadata['task_types']
start_vol = metadata['start_vol']
frame_times = np.arange(img.shape[-1]) * tr

# Check timepoint identity before loading the full image
assert np.isfinite(tr) and tr > 0, 'Invalid TR.'
assert isinstance(start_vol, int) and start_vol >= 0
assert img.shape[-1] == metadata['n_scans'] == len(design_mat), 'Image/design length mismatch.'
assert np.isclose(img.header.get_zooms()[3], tr, rtol=0, atol=1e-5), 'Image/design TR mismatch.'
assert design_mat.index.is_unique, 'Repeated frame times.'
assert np.allclose(design_mat.index.to_numpy(dtype=float), frame_times, rtol=0, atol=1e-5), 'Frame times must start at the first retained volume.'
assert list(design_mat.columns) == design_columns, 'Design columns differ from the metadata.'
assert design_mat.columns.is_unique, 'Repeated regressor names.'
assert np.isfinite(design_mat.to_numpy(dtype=float)).all(), 'Non-finite design values.'
assert task_regressors and set(task_regressors).issubset(design_mat.columns), 'Missing task regressors.'

if 'total_volumes' in metadata:
    assert img.shape[-1] == metadata['total_volumes'] - start_vol, 'Unexpected retained volume count.'
if 'frame_time_origin' in metadata:
    assert metadata['frame_time_origin'] == 'first_retained_volume'
if 'onset_shift_seconds' in metadata:
    assert np.isclose(metadata['onset_shift_seconds'], start_vol * tr)

# Outlier indices refer to the retained series
if 'outlier_volumes_trimmed' in metadata:
    outlier_volumes = np.asarray(metadata['outlier_volumes_trimmed'], dtype=int)
    assert np.all((outlier_volumes >= 0) & (outlier_volumes < len(design_mat)))
    assert len(np.unique(outlier_volumes)) == len(outlier_volumes)
    for i, volume_idx in enumerate(outlier_volumes):
        expected_spike = np.zeros(len(design_mat))
        expected_spike[volume_idx] = 1
        assert np.array_equal(design_mat[f'motion outlier#{i+1}'].to_numpy(), expected_spike), 'Spike indices differ from the metadata.'

data = img.get_fdata(dtype=np.float32)
print('Shared input:', metadata['fmri_path'])
print('Shape:', data.shape, '| TR:', tr, 's')
print('Initial volumes already removed:', start_vol)
print('Task regressors:', task_regressors)

In [ ]:
vol_shape = data.shape[:-1]  # Shape of each volume
n_vols = data.shape[-1]  # Number of retained timepoints

### Preprocessing

Each timepoint is one sample and each masked voxel is one feature at the PCA input. The intermediate matrix is stored as voxels x timepoints and transposed when passed to PCA. Background voxels are excluded before flattening the spatial dimensions. The same mask and voxel order are used to reconstruct the spatial loading maps.

Inspect the intensity distribution of the first retained volume before choosing a background threshold.

In [ ]:
plt.hist(data[:,:,:,0].ravel(), bins=100)
plt.xlabel("Voxel intensity")
plt.ylabel("Number of voxels")
plt.show()

Inspect the histogram on the shared preprocessed input. The foreground threshold is 500. Verify that it separates low-intensity background from tissue: intensity scales vary and this is not an anatomical brain extraction.


In [ ]:
# Select foreground voxels from the first retained volume
slice_non_background = data[:, :, :, 0] > 500
z = data.shape[2] // 2

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.imshow(data[:, :, z, 0].T, cmap='gray', origin='lower')
plt.title('Preprocessed fMRI')
plt.subplot(1, 2, 2)
plt.imshow(slice_non_background[:, :, z].T, cmap='gray', origin='lower')
plt.title('Mask (> 500)')
plt.tight_layout()
plt.savefig(op.join(save_results, 'pca_mask_check.png'), dpi=200, bbox_inches='tight')
plt.show()

Inspect the image and mask before continuing. Check that relevant tissue is retained and the background is excluded on the current preprocessed input. The intensity threshold is an empirical choice, not an anatomical brain extraction, and need not match the GLM's automatically estimated mask.

In [ ]:
# Rows are masked voxels; columns are timepoints
assert slice_non_background.any(), 'Empty mask: inspect the intensity threshold.'
samples = data[slice_non_background]
# Convert only the masked array for reliable temporal centering checks.
samples = samples.astype(np.float64, copy=False)
assert samples.shape == (int(slice_non_background.sum()), n_vols)
assert np.isfinite(samples).all(), 'Masked fMRI contains NaN/Inf.'
print('Masked voxels x timepoints:', samples.shape)


Subtract each voxel's temporal mean. Since `samples` is V x T, `axis=1` averages across time. Replicate those means across `n_vols` with `np.outer`, then subtract them.

This removes each voxel's static baseline, not the spatial mean of every volume. Time-varying spatially uniform fluctuations remain available to PCA. No voxel-wise variance standardization or GLM confound regression is applied. The GLM and PCA use the same retained observations and time reference, but their masks and signal transformations need not be identical.


In [ ]:
# Calculate each voxel's temporal mean across columns.
temporal_means = np.mean(samples, axis=1)
row_means = np.outer(temporal_means, np.ones(n_vols))
X = samples - row_means
assert np.isfinite(X).all()
assert np.allclose(X.mean(axis=1), 0, rtol=0, atol=1e-8), 'Voxel temporal means are not zero.'
print('Maximum absolute voxel temporal mean:', np.abs(X.mean(axis=1)).max())


In [ ]:
X.shape

`X` has shape `(n_masked_voxels, n_vols)`. Each row is a voxel timecourse relative to its temporal mean. `X.T` has shape `(n_vols, n_masked_voxels)` and is the matrix passed to PCA: one timepoint per sample, one masked voxel per feature.


### Fit PCA


Fit PCA on `X.T`. Components are continuous spatial loading patterns, not clusters or GLM activation maps. Retain up to 30 to explore the scree plot and cumulative explained variance. At least ten must remain available for the assignment's score-regressor correlations, even when fewer spatial maps are selected. `random_state=0` makes the fit reproducible when the automatically selected solver is randomized.


In [ ]:
# Fit with timepoints as samples and masked voxels as features.
nb_components = min(30, n_vols - 1, samples.shape[0])
if nb_components < 10:
    raise ValueError('At least ten components are needed for this analysis.')
assert X.T.shape == (n_vols, int(slice_non_background.sum()))

pca = PCA(n_components=nb_components, random_state=0)
pca.fit(X.T)
assert pca.components_.shape == (nb_components, samples.shape[0])
assert np.isfinite(pca.explained_variance_ratio_).all(), 'Undefined explained variance.'
print(pca.explained_variance_ratio_)


## Part 2: Select and display spatial components


In [ ]:
# Inspect the explained variance of successive components
explained_variance = pca.explained_variance_ratio_
plt.figure(figsize=(8, 5))
plt.plot(range(1, len(explained_variance) + 1), 100 * explained_variance, marker='o')
plt.xlabel('Principal component')
plt.ylabel('Explained variance (%)')
plt.title('PCA scree plot')
plt.xticks(range(1, len(explained_variance) + 1))
plt.grid(True)
plt.tight_layout()
plt.savefig(op.join(save_results, 'pca_scree.png'), dpi=200, bbox_inches='tight')
plt.show()

Four components are an initial display choice. Confirm or revise this number using the current scree plot, cumulative explained variance and spatial interpretability. Explain any change in slope and quote the actual retained percentage; the fact that earlier components explain more variance is not sufficient justification. The assignment does not impose a 95% variance target. Temporal correlations still use the first ten components.

In [ ]:
# Select the number of spatial maps to display
n_selected_components = 4
assert 1 <= n_selected_components <= pca.n_components_

ratios = pca.explained_variance_ratio_
cumulative_ratios = np.cumsum(ratios)
component_numbers = np.arange(1, len(ratios) + 1)
fig, ax = plt.subplots(1, 2, figsize=(12, 4))

ax[0].plot(component_numbers, 100 * ratios, 'o-')
ax[0].axvline(n_selected_components, color='red', linestyle='--', label='Selected maps')
ax[0].set_ylabel('Explained variance (%)')
ax[0].set_title('Scree plot')

ax[1].plot(component_numbers, 100 * cumulative_ratios, 'o-')
ax[1].axvline(n_selected_components, color='red', linestyle='--', label='Selected maps')
ax[1].set_ylabel('Cumulative explained variance (%)')
ax[1].set_title('Cumulative variance')

for k in range(2):
    ax[k].set_xlabel('Number of components')
    ax[k].legend()
    ax[k].grid(True, alpha=0.3)

print(f'First {n_selected_components} PCs: {100 * cumulative_ratios[n_selected_components - 1]:.2f}% of total variance')
fig.tight_layout()
fig.savefig(op.join(save_results, 'pca_variance_selection.png'), dpi=200, bbox_inches='tight')
plt.show()

pd.DataFrame({
    'component': component_numbers,
    'explained_variance_ratio': ratios,
    'cumulative_explained_variance_ratio': cumulative_ratios,
}).to_csv(op.join(save_results, 'pca_explained_variance.csv'), index=False)

Reconstruct each selected loading vector on the original image grid, using the same mask and voxel order. The fitted PCA is kept intact so that maps and temporal scores refer to the same directions. Signed loadings represent a component's spatial pattern, not statistical significance or hard cluster assignments.

In [ ]:
# Project the timepoints onto the first ten fitted spatial directions
scores = pca.transform(X.T)[:, :10]
assert scores.shape == (n_vols, 10)
assert np.isfinite(scores).all()

In [ ]:
# Put the selected loading vectors back into the image grid
pca_clusters = []
for component in pca.components_[:n_selected_components]:
    volume = np.zeros(vol_shape, dtype=np.float32)
    volume[slice_non_background] = component
    pca_clusters.append(volume)

print('Selected spatial maps:', len(pca_clusters))
print('Spatial shape:', pca_clusters[0].shape)

In [ ]:
# Show each spatial loading map separately in axial view
mean_fmri = nib.Nifti1Image(data.mean(axis=-1), affine, img.header.copy())
common_vmax = float(np.max(np.abs(pca.components_[:n_selected_components])))

for visual_idx in range(n_selected_components):
    component_img = nib.Nifti1Image(pca_clusters[visual_idx], affine, img.header.copy())
    component_img.set_data_dtype(np.float32)
    component_img.to_filename(op.join(save_results, f'PC{visual_idx + 1:02d}_loadings.nii.gz'))

    # Apply a fixed display threshold; this is not a significance threshold.
    visual_threshold = 1e-3
    component_display = plot_stat_map(
        component_img, bg_img=mean_fmri, display_mode='z', cut_coords=5,
        threshold=visual_threshold, cmap='RdBu_r', symmetric_cbar=True,
        vmax=common_vmax,
        title=f'PC{visual_idx + 1} | variance {100 * ratios[visual_idx]:.2f}%',
    )
    component_display.savefig(op.join(save_results, f'PC{visual_idx + 1:02d}_axial.png'), dpi=200)
    plt.show()
    component_display.close()

## Part 3: Correlation with the GLM regressors

Selected spatial maps and the first ten score timecourses are taken from the same fitted model, without a second fit. Each selected loading map is displayed separately in axial view.

Compare the first ten temporal scores with the exact exported design, including HRF-convolved task regressors and nuisance columns. Constant columns are excluded because Pearson correlation is undefined for them. Do not convolve the BOLD-derived scores with another HRF. Regressors enter only this interpretation step, not the PCA fit.

Correlations are descriptive marginal associations, not partial GLM effects or corrected activation tests. Correlated task regressors may share the same association. Motion, drift and spike associations help identify nuisance-dominated components; these effects have not been regressed out of the PCA input. Temporal autocorrelation prevents treating all scans as independent observations for naive significance tests.

A spatial map and its score can both reverse sign without changing the reconstruction. Interpret the sign together with the map, rather than labelling a component as activation or deactivation from the score alone.

In [ ]:
# Keep score rows on the same frame-time index as the GLM design
assert scores.shape[0] == len(design_mat) == n_vols
score_columns = [f'PC{i+1}' for i in range(10)]
score_df = pd.DataFrame(scores, index=design_mat.index, columns=score_columns)
score_df.index.name = 'frame_time_s'

# Exclude constant columns only from the correlation calculation
regressor_columns = [c for c in design_mat.columns if design_mat[c].std(ddof=0) > 1e-12]
excluded_columns = [c for c in design_mat.columns if c not in regressor_columns]
print('Constant columns excluded from Pearson r:', excluded_columns)
assert regressor_columns, 'No variable GLM regressors.'
assert (score_df.std(ddof=0) > 1e-12).all(), 'A component score has zero variance.'

# Compare temporal scores with task, motion, drift and spike regressors
pca_regressor_corr = pd.DataFrame(index=score_columns, columns=regressor_columns, dtype=float)
for component in score_columns:
    for regressor in regressor_columns:
        pca_regressor_corr.loc[component, regressor] = score_df[component].corr(design_mat[regressor])

assert np.isfinite(pca_regressor_corr.to_numpy()).all(), 'Undefined correlation.'
assert (np.abs(pca_regressor_corr.to_numpy()) <= 1 + 1e-12).all()

plt.figure(figsize=(max(10, 0.55 * len(regressor_columns)), 6))
plt.imshow(pca_regressor_corr, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
plt.colorbar(label='Pearson r')
plt.xticks(range(len(regressor_columns)), regressor_columns, rotation=90, fontsize=8)
plt.yticks(range(10), score_columns)
plt.title('First 10 PCA temporal scores vs GLM regressors')
plt.tight_layout()
plt.savefig(op.join(save_results, 'pca_regressor_correlations.png'), dpi=200, bbox_inches='tight')
plt.show()

score_df.to_csv(op.join(save_results, 'pca_scores_first10.csv'))
pca_regressor_corr.to_csv(op.join(save_results, 'pca_regressor_correlations.csv'))
pca_regressor_corr

### Relate temporal associations to the spatial maps

For each selected component, report its explained variance, the strongest task associations and the main regions visible in its loading map. Check whether a stronger association with motion, drift or a spike suggests a nuisance contribution. Inspect the loading map of a component beyond the displayed set before making anatomical claims about it.

Complete the interpretation after running the subject data: **PC ... explains ...% of total variance and correlates with ... (r = ...). Its loading map shows prominent weights in ... . The association with ... limits/supports a task-related interpretation because ... .** The table below is a descriptive aid, not a significance test. Four displayed maps and the intensity threshold remain choices to justify from the current figures.

In [ ]:
# Summarize the strongest task and nuisance association for each score
task_columns = [c for c in task_regressors if c in regressor_columns]
nuisance_columns = [c for c in regressor_columns if c not in task_regressors]
assert task_columns, 'No variable task regressors are available.'

summary_rows = []
for component in score_columns:
    best_task = pca_regressor_corr.loc[component, task_columns].abs().idxmax()
    row = {
        'PC': component,
        'explained_variance_percent': 100 * pca.explained_variance_ratio_[score_columns.index(component)],
        'strongest_task': best_task,
        'task_r': pca_regressor_corr.loc[component, best_task],
    }
    if nuisance_columns:
        best_nuisance = pca_regressor_corr.loc[component, nuisance_columns].abs().idxmax()
        row['strongest_nuisance'] = best_nuisance
        row['nuisance_r'] = pca_regressor_corr.loc[component, best_nuisance]
    summary_rows.append(row)

pca_task_summary = pd.DataFrame(summary_rows)
pca_task_summary.to_csv(op.join(save_results, 'pca_task_summary.csv'), index=False)
pca_task_summary.round(3)

### Compare with the GLM spatial results

Temporal correlations require only the design matrix; anatomical comparison additionally requires the fitted GLM maps. Place `glm_story_z.nii.gz`, `glm_math_z.nii.gz` and `glm_story_minus_math_z.nii.gz` in `shared_dir` when they are available. The next cell displays those maps without a statistical threshold, for descriptive inspection. Use the inferential thresholds and correction chosen in Part 1 before calling a region significantly activated.

Compare labelled axial coordinates and visible anatomical landmarks with the PCA maps. A common array shape or a matching filename does not establish anatomical registration. For a figure showing direct correspondence, select the same axial world coordinates in both plots after verifying their common spatial reference. PCA loading magnitudes and GLM z scores have different units and must not share a numerical interpretation.

If the GLM includes both whole blocks and their nested sub-events, its story/math coefficients are conditional on the other regressors. They are not automatically the overall task effects of a block-only model. Similar temporal associations therefore need a cautious spatial interpretation.

In [ ]:
# Inspect available GLM maps without interrupting the temporal analysis
glm_map_names = ['story', 'math', 'story_minus_math']
for map_name in glm_map_names:
    glm_map_path = op.join(shared_dir, f'glm_{map_name}_z.nii.gz')
    if op.isfile(glm_map_path):
        glm_display = plot_stat_map(
            glm_map_path, bg_img=mean_fmri, display_mode='z', cut_coords=5,
            threshold=0, cmap='RdBu_r', symmetric_cbar=True,
            title=f'GLM {map_name}: unthresholded z scores',
        )
        glm_display.savefig(op.join(save_results, f'glm_{map_name}_axial.png'), dpi=200)
        plt.show()
        glm_display.close()
    else:
        print('GLM map not available:', glm_map_path)

## Theoretical questions

**1. What changes if one voxel is one sample?**

With timepoints as samples, the matrix is T x V: each spatial component has V weights and T temporal scores. With voxels as samples, it becomes V x T: each component has T weights describing a temporal pattern, and V scores describing its expression across voxels. The analysis would then summarize differences between voxel timecourses. Centering would also act along a different sample axis, so transposing the previous outputs alone would not reproduce the new analysis.

**2. Is comparison with the GLM conceptually appropriate?**

The methods provide complementary descriptions of the same observations when the retained timepoints and regressors are aligned. A GLM tests associations with a design specified in advance and supports task contrasts voxel by voxel. PCA finds orthogonal patterns that maximize explained variance without using the task labels. Correlating temporal scores with the design helps interpret these patterns; comparing loading maps with GLM maps addresses their spatial distribution. A correlation does not establish equivalence with a GLM effect because a component can mix task, motion and physiological variability. We would use GLM for predefined hypotheses and inferential contrasts, and PCA for exploration, dimensionality reduction and identifying dominant patterns or artefacts.

## Preparing the shared files in the GLM session

Run the following in the GLM notebook after building `design_mat`, with `fmri_img` loaded from the preprocessed series actually used by the model. Use the same `data_dir` selected in the PCA setup cell. The export is stored under `data_dir/outputs/shared_glm_pca` on the shared drive. It does not fit PCA or rebuild the GLM design. If `make_first_level_design_matrix` is missing from the GLM imports, import it from `nilearn.glm.first_level` before building the design.

```python
import json

shared_dir = op.join(data_dir, 'outputs', 'shared_glm_pca')
os.makedirs(shared_dir, exist_ok=True)
assert len(design_mat) == fmri_img.shape[-1]
assert np.allclose(design_mat.index.to_numpy(dtype=float),
                   np.arange(len(design_mat)) * tr, rtol=0, atol=1e-5)
design_mat.to_csv(op.join(shared_dir, 'design_matrix.csv'),
                  index_label='frame_time_s')
metadata = {
    'fmri_path': op.abspath(fmri_img.get_filename()),
    'tr_s': float(tr),
    'start_vol': int(start_vol),
    'n_scans': int(fmri_img.shape[-1]),
    'columns': list(design_mat.columns),
    'task_regressors': sorted(events['trial_type'].unique().tolist()),
}
with open(op.join(shared_dir, 'metadata.json'), 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2)
```

After fitting the GLM successfully, the following optional export supplies the maps used in the spatial comparison. The contrast weights follow the actual fitted design column order. These exports are unthresholded z maps, not corrected activation results.

```python
fitted_design = fit_glm.design_matrices_[0]
columns = list(fitted_design.columns)
assert columns == list(design_mat.columns)
assert np.allclose(fitted_design.to_numpy(), design_mat.to_numpy())
story_contrast = np.zeros(len(columns))
math_contrast = np.zeros(len(columns))
story_contrast[columns.index('story')] = 1
math_contrast[columns.index('math')] = 1
contrasts = {
    'story': story_contrast,
    'math': math_contrast,
    'story_minus_math': story_contrast - math_contrast,
}
for name, contrast in contrasts.items():
    z_map = fit_glm.compute_contrast(contrast, output_type='z_score')
    z_map.to_filename(op.join(shared_dir, f'glm_{name}_z.nii.gz'))
```

Use the acquisition TR and the exported frame-time grid. If paths or upstream GLM choices change, regenerate the shared exports in the VM. Subject variance values, mask quality and anatomical conclusions must come from execution on the subject data.